In [6]:
"""
Comprehensive Data Collection Script
Collects People, Events, Discoveries, and Places with multilingual support
"""

from src.data import (
    EntityDatabase,
    WikidataClient,
    WikipediaClient,
    WESTERN_COUNTRIES,
    ARABIC_COUNTRIES,
    ASIAN_COUNTRIES,
)
from src.utils import setup_logging
import logging
import json
import time

# Setup logging
setup_logging(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize clients
db = EntityDatabase("data/entities.db")
wikidata = WikidataClient(rate_limit_delay=1.0)
wikipedia = WikipediaClient(language='en', rate_limit_delay=0.1)


def has_complete_labels(entity, entity_type_key='person'):
    """
    Check if entity has complete labels in en, fr, ar
    Arabic can be either 'ar' or 'arz' (fallback)
    """
    required_keys = [
        f'{entity_type_key}Label_en',
        f'{entity_type_key}Label_fr',
    ]

    # Check en and fr
    for key in required_keys:
        if not entity.get(key):
            return False

    # Check ar or arz for Arabic
    ar_key = f'{entity_type_key}Label_ar'
    if not entity.get(ar_key):
        return False  # Arabic fallback already applied by wikidata_client

    return True


def collect_people(nationalities, limit=100, min_year=1850, max_year=2000):
    """
    Collect people entities with full metadata

    Note: Results are in random order (ORDER BY causes timeout).
    Max limit: 10000 per query.

    For larger collections, split by nationality or year range:
    - Example 1: Query each nationality separately
    - Example 2: Split year range into chunks (1850-1900, 1900-1950, etc.)
    """
    logger.info(f"Collecting up to {limit} people...")

    # Query Wikidata (fast, no ORDER BY)
    people = wikidata.get_people_multilingual(
        nationalities=nationalities,
        limit=limit,
        min_year=min_year,
        max_year=max_year,
    )

    logger.info(f"Got {len(people)} people from Wikidata")

    # Labels are now fetched in SPARQL query (MUCH faster than API calls!)
    logger.info("Step 2/5: Labels already included in SPARQL query (skipping API calls)")

    logger.info("Step 3/5: Validating entities (checking for missing fields)...")
    # Log entities with missing fields BEFORE filtering
    filtered_count = 0
    for person in people:
        wikidata_id = person.get('person_id', 'UNKNOWN')
        missing_fields = []

        # Check required labels (only person labels, others are optional now)
        if not person.get('personLabel_en'):
            missing_fields.append('personLabel_en')
        if not person.get('personLabel_fr'):
            missing_fields.append('personLabel_fr')
        if not person.get('personLabel_ar'):
            missing_fields.append('personLabel_ar (no ar or arz)')

        # Check required facts (IDs only, not labels)
        if not person.get('birthDate'):
            missing_fields.append('birthDate')
        if not person.get('birthPlace'):
            missing_fields.append('birthPlace')

        # Note: birthPlace/nationality/occupation LABELS are optional
        # We only need the IDs (URLs) which are already present

        if missing_fields:
            filtered_count += 1
            person_name = person.get('personLabel_en') or person.get('personLabel_fr') or wikidata_id
            logger.warning(
                f"Entity {wikidata_id} ({person_name}) missing fields: {', '.join(missing_fields)} "
                f"- Will be filtered out. Check: https://www.wikidata.org/wiki/{wikidata_id}"
            )

    if filtered_count > 0:
        logger.info(f"Found {filtered_count} entities with missing fields (will be filtered)")

    # Filter for complete labels
    logger.info("Step 4/5: Filtering entities with complete labels...")
    valid_people = [p for p in people if has_complete_labels(p, 'person')]
    logger.info(f"Filtered to {len(valid_people)} entities with complete labels ({len(people) - len(valid_people)} removed)")

    logger.info("Step 5/5: Fetching Wikipedia texts and storing in database...")
    collected = 0
    for i, person in enumerate(valid_people, 1):
        # Progress indicator every 10 entities
        if i % 10 == 0 or i == len(valid_people):
            logger.info(f"  Processing entity {i}/{len(valid_people)} ({i*100//len(valid_people)}%, {collected} stored so far)")

        wikidata_id = person.get('person_id')
        if not wikidata_id:
            continue

        # Determine culture from nationality
        nationality_id = person.get('nationality_id')
        if nationality_id in WESTERN_COUNTRIES:
            culture = 'Western'
        elif nationality_id in ARABIC_COUNTRIES:
            culture = 'Arabic'
        elif nationality_id in ASIAN_COUNTRIES:
            culture = 'Asian'
        else:
            culture = 'Other'

        # Add base entity
        entity_id = db.add_entity(
            wikidata_id=wikidata_id,
            entity_type='person',
            culture=culture,
            country=person.get('nationalityLabel_en')
        )

        if not entity_id:
            continue

        # Add labels in all languages
        for lang in ['en', 'fr', 'ar']:
            label = person.get(f'personLabel_{lang}')
            if label:
                db.add_entity_label(entity_id, lang, label)

        # Add metadata
        if person.get('gender'):
            db.add_metadata(entity_id, 'gender', person['gender'])

        if person.get('occupation'):
            # Store all occupations as JSON list
            db.add_metadata(entity_id, 'occupation', json.dumps([person['occupation']]))

        if person.get('birthDate'):
            birth_year = person['birthDate'][:4]  # Extract year
            db.add_metadata(entity_id, 'birth_year', birth_year)

        if person.get('deathDate'):
            death_year = person['deathDate'][:4]
            db.add_metadata(entity_id, 'death_year', death_year)
            db.add_metadata(entity_id, 'is_alive', 'False')
        else:
            db.add_metadata(entity_id, 'is_alive', 'True')

        # Fetch Wikipedia language count separately (faster than SPARQL)
        num_wiki_langs = wikidata.get_sitelinks_count(wikidata_id)
        db.add_metadata(entity_id, 'num_wikipedia_languages', str(num_wiki_langs))

        # Get exact Wikipedia titles from Wikidata (more reliable than searching!)
        wikipedia_titles = wikidata.get_wikipedia_titles(wikidata_id)

        # Fetch Wikipedia using exact titles
        wiki_pages = {}
        for lang in ['en', 'fr', 'ar']:
            if lang in wikipedia_titles:
                wikipedia.change_language(lang)
                page_data = wikipedia.get_page_by_title(wikipedia_titles[lang])
                wiki_pages[lang] = page_data
            else:
                logger.info(f"No {lang} Wikipedia sitelink for {wikidata_id}")
                wiki_pages[lang] = None

        # Add facts (birth date, birth place, nationality)
        # Birth date
        if person.get('birthDate'):
            fact_id = db.add_fact(entity_id, 'P569', person['birthDate'])

        # Birth place (URL + multilingual labels)
        if person.get('birthPlace'):
            fact_id = db.add_fact(entity_id, 'P19', person['birthPlace'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'birthPlaceLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Nationality
        if person.get('nationality'):
            fact_id = db.add_fact(entity_id, 'P27', person['nationality'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'nationalityLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Track which languages have Wikipedia
        has_wikipedia = []
        for lang, page_data in wiki_pages.items():
            if page_data:
                has_wikipedia.append(lang)

                # Extract metadata
                metadata = page_data.get('metadata', {})

                # Store Wikipedia text with metadata
                db.add_wikipedia_text(
                    entity_id=entity_id,
                    language=lang,
                    title=page_data.get('title'),
                    summary=page_data.get('summary'),
                    content=page_data.get('content'),
                    url=page_data.get('url'),
                    text_length_chars=metadata.get('text_length_chars'),
                    text_length_words=metadata.get('text_length_words'),
                    num_sections=metadata.get('num_sections'),
                )

        # Store Wikipedia availability metadata
        db.add_metadata(entity_id, 'wikipedia_completeness', str(len(has_wikipedia)))
        db.add_metadata(entity_id, 'wikipedia_languages', json.dumps(has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_en', str('en' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_fr', str('fr' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_ar', str('ar' in has_wikipedia))
        db.add_metadata(entity_id, 'has_all_wikipedia_languages', str(len(has_wikipedia) == 3))

        collected += 1

    logger.info(f"✅ Completed! Collected {collected}/{len(valid_people)} entities successfully")
    return collected


def collect_events(event_types=None, limit=50, min_year=1800, max_year=2000):
    """Collect event entities with full metadata"""
    logger.info(f"Collecting {limit} events...")

    # Query Wikidata
    events = wikidata.get_events_multilingual(
        event_types=event_types,
        limit=limit,
        min_year=min_year,
        max_year=max_year
    )

    logger.info(f"Got {len(events)} events from Wikidata")

    # Filter for complete labels
    valid_events = [e for e in events if has_complete_labels(e, 'event')]
    logger.info(f"Filtered to {len(valid_events)} with complete labels")

    collected = 0
    for event in valid_events:
        wikidata_id = event.get('event_id')
        if not wikidata_id:
            continue

        # Add base entity
        entity_id = db.add_entity(
            wikidata_id=wikidata_id,
            entity_type='event',
            culture=None,  # Events don't have inherent culture
            country=event.get('locationLabel_en')  # Use location as country
        )

        if not entity_id:
            continue

        # Add labels
        for lang in ['en', 'fr', 'ar']:
            label = event.get(f'eventLabel_{lang}')
            if label:
                db.add_entity_label(entity_id, lang, label)

        # Add metadata
        if event.get('eventType'):
            db.add_metadata(entity_id, 'event_type', event['eventType'])

        if event.get('date'):
            event_year = event['date'][:4]
            db.add_metadata(entity_id, 'event_year', event_year)

        # Fetch Wikipedia language count separately
        num_wiki_langs = wikidata.get_sitelinks_count(wikidata_id)
        db.add_metadata(entity_id, 'num_wikipedia_languages', str(num_wiki_langs))

        # Add facts
        if event.get('date'):
            db.add_fact(entity_id, 'P585', event['date'])

        if event.get('location'):
            fact_id = db.add_fact(entity_id, 'P276', event['location'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = event.get(f'locationLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Fetch Wikipedia
        entity_names = {
            'en': event.get('eventLabel_en'),
            'fr': event.get('eventLabel_fr'),
            'ar': event.get('eventLabel_ar')
        }

        wiki_pages = wikipedia.get_page_multilingual(entity_names)

        has_wikipedia = []
        for lang, page_data in wiki_pages.items():
            if page_data:
                has_wikipedia.append(lang)
                metadata = page_data.get('metadata', {})

                db.add_wikipedia_text(
                    entity_id=entity_id,
                    language=lang,
                    title=page_data.get('title'),
                    summary=page_data.get('summary'),
                    content=page_data.get('content'),
                    url=page_data.get('url'),
                    text_length_chars=metadata.get('text_length_chars'),
                    text_length_words=metadata.get('text_length_words'),
                    num_sections=metadata.get('num_sections'),
                    num_references=metadata.get('num_references'),
                    num_backlinks=metadata.get('num_backlinks'),
                )

        # Store Wikipedia availability
        db.add_metadata(entity_id, 'wikipedia_completeness', str(len(has_wikipedia)))
        db.add_metadata(entity_id, 'wikipedia_languages', json.dumps(has_wikipedia))
        db.add_metadata(entity_id, 'has_all_wikipedia_languages', str(len(has_wikipedia) == 3))

        collected += 1
        logger.info(f"Collected event {collected}/{len(valid_events)}: {entity_names['en']}")

    return collected

2025-12-16 23:56:02,721 - src.utils - INFO - Logging initialized
2025-12-16 23:56:02,758 - src.data.database - INFO - Database initialized at data/entities.db


In [7]:

def collect_people(nationalities, limit=100, min_year=1850, max_year=2000):
    """
    Collect people entities with full metadata

    Note: Results are in random order (ORDER BY causes timeout).
    Max limit: 10000 per query.

    For larger collections, split by nationality or year range:
    - Example 1: Query each nationality separately
    - Example 2: Split year range into chunks (1850-1900, 1900-1950, etc.)
    """
    logger.info(f"Collecting up to {limit} people...")

    # Query Wikidata (fast, no ORDER BY)
    people = wikidata.get_people_multilingual(
        nationalities=nationalities,
        limit=limit,
        min_year=min_year,
        max_year=max_year,
    )

    logger.info(f"Got {len(people)} people from Wikidata (may include duplicates)")

    # Step 2: Deduplicate by person ID (person can have multiple occupations/nationalities)
    logger.info("Step 2/5: Deduplicating by person ID...")
    seen_ids = set()
    unique_people = []
    for person in people:
        person_id = person['person'].split('/')[-1]
        if person_id not in seen_ids:
            seen_ids.add(person_id)
            unique_people.append(person)

    duplicates_removed = len(people) - len(unique_people)
    logger.info(f"Deduplicated to {len(unique_people)} unique people ({duplicates_removed} duplicates removed)")
    people = unique_people

    # Labels are now fetched in SPARQL query (MUCH faster than API calls!)
    logger.info("Step 3/5: Labels already included in SPARQL query (skipping API calls)")

    logger.info("Step 4/5: Validating entities (checking for missing fields)...")
    # Log entities with missing fields BEFORE filtering
    filtered_count = 0
    for person in people:
        wikidata_id = person.get('person_id', 'UNKNOWN')
        missing_fields = []

        # Check required labels (only person labels, others are optional now)
        if not person.get('personLabel_en'):
            missing_fields.append('personLabel_en')
        if not person.get('personLabel_fr'):
            missing_fields.append('personLabel_fr')
        if not person.get('personLabel_ar'):
            missing_fields.append('personLabel_ar (no ar or arz)')

        # Check required facts (IDs only, not labels)
        if not person.get('birthDate'):
            missing_fields.append('birthDate')
        if not person.get('birthPlace'):
            missing_fields.append('birthPlace')

        # Note: birthPlace/nationality/occupation LABELS are optional
        # We only need the IDs (URLs) which are already present

        if missing_fields:
            filtered_count += 1
            person_name = person.get('personLabel_en') or person.get('personLabel_fr') or wikidata_id
            logger.warning(
                f"Entity {wikidata_id} ({person_name}) missing fields: {', '.join(missing_fields)} "
                f"- Will be filtered out. Check: https://www.wikidata.org/wiki/{wikidata_id}"
            )

    if filtered_count > 0:
        logger.info(f"Found {filtered_count} entities with missing fields (will be filtered)")

    # Filter for complete labels
    logger.info("Step 5/5: Filtering entities with complete labels...")
    valid_people = [p for p in people if has_complete_labels(p, 'person')]
    logger.info(f"Filtered to {len(valid_people)} entities with complete labels ({len(people) - len(valid_people)} removed)")

    logger.info("Step 6/6: Fetching Wikipedia texts and storing in database...")
    collected = 0
    for i, person in enumerate(valid_people, 1):
        # Progress indicator every 10 entities
        if i % 10 == 0 or i == len(valid_people):
            logger.info(f"  Processing entity {i}/{len(valid_people)} ({i*100//len(valid_people)}%, {collected} stored so far)")

        wikidata_id = person.get('person_id')
        if not wikidata_id:
            continue

        # Determine culture from nationality
        nationality_id = person.get('nationality_id')
        if nationality_id in WESTERN_COUNTRIES:
            culture = 'Western'
        elif nationality_id in ARABIC_COUNTRIES:
            culture = 'Arabic'
        elif nationality_id in ASIAN_COUNTRIES:
            culture = 'Asian'
        else:
            culture = 'Other'

        # Add base entity
        entity_id = db.add_entity(
            wikidata_id=wikidata_id,
            entity_type='person',
            culture=culture,
            country=person.get('nationalityLabel_en')
        )

        if not entity_id:
            continue

        # Add labels in all languages
        for lang in ['en', 'fr', 'ar']:
            label = person.get(f'personLabel_{lang}')
            if label:
                db.add_entity_label(entity_id, lang, label)

        # Add metadata
        if person.get('gender'):
            db.add_metadata(entity_id, 'gender', person['gender'])

        if person.get('occupation'):
            # Store all occupations as JSON list
            db.add_metadata(entity_id, 'occupation', json.dumps([person['occupation']]))

        if person.get('birthDate'):
            birth_year = person['birthDate'][:4]  # Extract year
            db.add_metadata(entity_id, 'birth_year', birth_year)

        if person.get('deathDate'):
            death_year = person['deathDate'][:4]
            db.add_metadata(entity_id, 'death_year', death_year)
            db.add_metadata(entity_id, 'is_alive', 'False')
        else:
            db.add_metadata(entity_id, 'is_alive', 'True')

        # Fetch Wikipedia language count separately (faster than SPARQL)
        num_wiki_langs = wikidata.get_sitelinks_count(wikidata_id)
        db.add_metadata(entity_id, 'num_wikipedia_languages', str(num_wiki_langs))

        # Get exact Wikipedia titles from Wikidata (more reliable than searching!)
        wikipedia_titles = wikidata.get_wikipedia_titles(wikidata_id)

        # Fetch Wikipedia using exact titles
        wiki_pages = {}
        for lang in ['en', 'fr', 'ar']:
            if lang in wikipedia_titles:
                wikipedia.change_language(lang)
                page_data = wikipedia.get_page_by_title(wikipedia_titles[lang])
                wiki_pages[lang] = page_data
            else:
                logger.info(f"No {lang} Wikipedia sitelink for {wikidata_id}")
                wiki_pages[lang] = None

        # Add facts (birth date, birth place, nationality)
        # Birth date
        if person.get('birthDate'):
            fact_id = db.add_fact(entity_id, 'P569', person['birthDate'])

        # Birth place (URL + multilingual labels)
        if person.get('birthPlace'):
            fact_id = db.add_fact(entity_id, 'P19', person['birthPlace'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'birthPlaceLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Nationality
        if person.get('nationality'):
            fact_id = db.add_fact(entity_id, 'P27', person['nationality'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'nationalityLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Track which languages have Wikipedia
        has_wikipedia = []
        for lang, page_data in wiki_pages.items():
            if page_data:
                has_wikipedia.append(lang)

                # Extract metadata
                metadata = page_data.get('metadata', {})

                # Store Wikipedia text with metadata
                db.add_wikipedia_text(
                    entity_id=entity_id,
                    language=lang,
                    title=page_data.get('title'),
                    summary=page_data.get('summary'),
                    content=page_data.get('content'),
                    url=page_data.get('url'),
                    text_length_chars=metadata.get('text_length_chars'),
                    text_length_words=metadata.get('text_length_words'),
                    num_sections=metadata.get('num_sections'),
                )

        # Store Wikipedia availability metadata
        db.add_metadata(entity_id, 'wikipedia_completeness', str(len(has_wikipedia)))
        db.add_metadata(entity_id, 'wikipedia_languages', json.dumps(has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_en', str('en' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_fr', str('fr' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_ar', str('ar' in has_wikipedia))
        db.add_metadata(entity_id, 'has_all_wikipedia_languages', str(len(has_wikipedia) == 3))

        collected += 1

    logger.info(f"✅ Completed! Collected {collected}/{len(valid_people)} entities successfully")
    return collected

In [8]:
total_western = collect_people(
        nationalities=WESTERN_COUNTRIES,  # France, Germany, US, Spain
        limit=10000,  # Max 10000 works great
        min_year=1850,
        max_year=2000
    )
logger.info(f"Collected {total_western} Western people")

2025-12-16 23:56:06,191 - __main__ - INFO - Collecting up to 10000 people...
2025-12-16 23:56:06,193 - src.data.wikidata_client - INFO - Querying 4 nationalities (multilingual), limit=10000
2025-12-16 23:56:07,414 - src.data.wikidata_client - INFO - Query successful: 10000 results
2025-12-16 23:56:07,459 - __main__ - INFO - Got 10000 people from Wikidata (may include duplicates)
2025-12-16 23:56:07,459 - __main__ - INFO - Step 2/5: Deduplicating by person ID...
2025-12-16 23:56:07,462 - __main__ - INFO - Deduplicated to 4112 unique people (5888 duplicates removed)
2025-12-16 23:56:07,466 - __main__ - INFO - Step 3/5: Labels already included in SPARQL query (skipping API calls)
2025-12-16 23:56:07,467 - __main__ - INFO - Step 4/5: Validating entities (checking for missing fields)...
2025-12-16 23:56:07,467 - __main__ - WARNING - Entity Q31878 (Hermidita) missing fields: personLabel_ar (no ar or arz) - Will be filtered out. Check: https://www.wikidata.org/wiki/Q31878
2025-12-16 23:56:07,

KeyboardInterrupt: 

In [2]:
total_western = collect_people(
        nationalities=WESTERN_COUNTRIES,
        limit=5,
        offset=0,  # First 50 entities
        min_year=1850,
        max_year=2000,
        sort=False
    )
logger.info(f"Collected {total_western} Western people (batch 1)")

2025-12-16 22:20:19,336 - __main__ - INFO - Collecting 5 people (offset=0, sort=False)...
2025-12-16 22:20:19,336 - src.data.wikidata_client - INFO - Querying 4 nationalities (multilingual), limit=5, offset=0, sort=False
2025-12-16 22:20:36,049 - src.data.wikidata_client - INFO - Query successful: 5 results
2025-12-16 22:20:36,051 - __main__ - INFO - Got 5 people from Wikidata
2025-12-16 22:20:36,052 - __main__ - WARNING - Entity Q80310 (Ángel Otaegui) missing fields: personLabel_ar (no ar or arz) - Will be filtered out. Check: https://www.wikidata.org/wiki/Q80310
2025-12-16 22:20:36,052 - __main__ - INFO - Filtered to 4 with complete labels
2025-12-16 22:20:36,059 - src.data.database - INFO - Added entity: Q26853
2025-12-16 22:20:48,519 - src.data.wikipedia_client - INFO - Fetched Wikipedia page: Camilo Cienfuegos (44985 chars)
2025-12-16 22:20:50,169 - src.data.wikipedia_client - INFO - Fetched Wikipedia page: Camilo Cienfuegos (5275 chars)
2025-12-16 22:20:52,311 - src.data.wikipedi

In [5]:
total_western = collect_people(WESTERN_COUNTRIES, limit=10, offset=0)

2025-12-16 22:33:51,879 - __main__ - INFO - Collecting 10 people (offset=0)...
2025-12-16 22:33:51,879 - src.data.wikidata_client - INFO - Querying 4 nationalities (multilingual), limit=10, offset=0


KeyboardInterrupt: 

In [10]:
stats = db.get_statistics()

In [11]:
stats

{'total_entities': 53,
 'by_type': {'person': 53},
 'by_culture': {'Western': 53},
 'labels_by_language': {'ar': 53, 'en': 53, 'fr': 53},
 'wikipedia_by_language': {'ar': 50, 'en': 50, 'fr': 48},
 'total_questions': 0,
 'total_facts': 156}

In [12]:
# Get all entities
entities = db.get_all_entities()

print("=" * 60)
print("COLLECTED ENTITIES")
print("=" * 60)

for entity in entities:
    entity_id = entity['id']
    wikidata_id = entity['wikidata_id']

    # Get labels
    labels = db.get_entity_labels(entity_id)

    # Get metadata
    metadata = db.get_metadata(entity_id)

    # Get Wikipedia texts
    wiki_texts = db.get_wikipedia_texts(entity_id)

    print(f"\nEntity: {labels.get('en', 'N/A')}")
    print(f"  Wikidata ID: {wikidata_id}")
    print(f"  Culture: {entity['culture']}")
    print(f"  Labels:")
    print(f"    EN: {labels.get('en', 'Missing')}")
    print(f"    FR: {labels.get('fr', 'Missing')}")
    print(f"    AR: {labels.get('ar', 'Missing')}")

    print(f"  Metadata:")
    print(f"    Gender: {metadata.get('gender', 'N/A')}")
    print(f"    Birth year: {metadata.get('birth_year', 'N/A')}")
    print(f"    Wikipedia languages: {metadata.get('num_wikipedia_languages', '0')}")
    print(f"    Has all Wikipedia: {metadata.get('has_all_wikipedia_languages', 'False')}")

    print(f"  Wikipedia:")
    for lang in ['en', 'fr', 'ar']:
        if lang in wiki_texts:
            text = wiki_texts[lang]
            print(f"    {lang.upper()}: {text['title']} ({len(text['content'])} chars)")
        else:
            print(f"    {lang.upper()}: Missing")

    # Get facts
    facts = db.get_entity_facts(entity_id)
    if facts:
        print(f"  Facts: {len(facts)} properties")
        for fact in facts[:3]:  # Show first 3
            labels_dict = fact.get('labels', {})
            print(f"    {fact['property']}: {labels_dict.get('en', 'N/A')}")

print("\n" + "=" * 60)

COLLECTED ENTITIES

Entity: Francisco Franco
  Wikidata ID: Q29179
  Culture: Western
  Labels:
    EN: Francisco Franco
    FR: Francisco Franco
    AR: فرانثيسكو فرانكو
  Metadata:
    Gender: http://www.wikidata.org/entity/Q6581097
    Birth year: 1892
    Wikipedia languages: 117
    Has all Wikipedia: True
  Wikipedia:
    EN: Francisco Franco (93235 chars)
    FR: Francisco Franco (600994 chars)
    AR: فرانثيسكو فرانكو (55296 chars)
  Facts: 3 properties
    P19: Ferrol
    P27: Spain
    P569: N/A

Entity: Salva Sevilla
  Wikidata ID: Q54835
  Culture: Western
  Labels:
    EN: Salva Sevilla
    FR: Salva Sevilla
    AR: سالفا سيفييا
  Metadata:
    Gender: http://www.wikidata.org/entity/Q6581097
    Birth year: 1984
    Wikipedia languages: 24
    Has all Wikipedia: True
  Wikipedia:
    EN: Salva Sevilla (2967 chars)
    FR: Salva Sevilla (623 chars)
    AR: سالفا سيفييا (616 chars)
  Facts: 3 properties
    P19: Berja
    P27: Spain
    P569: N/A

Entity: Santiago Ramón y Ca

In [13]:
wikidata_id = "Q83006"  # David Villa
entity = db.get_entity(wikidata_id)
entity_id = entity['id']

# Get Wikipedia texts
wiki_texts = db.get_wikipedia_texts(entity_id)

print("=" * 80)
print(f"Wikipedia Articles for: {wikidata_id}")
print("=" * 80)

for lang, text_data in wiki_texts.items():
    print(f"\n{'='*80}")
    print(f"LANGUAGE: {lang.upper()}")
    print(f"{'='*80}")
    print(f"Title: {text_data['title']}")
    print(f"URL: {text_data['url']}")
    print(f"\nSummary:")
    print(text_data['summary'])
    print(f"\nFull Content ({len(text_data['content'])} chars):")
    print(text_data['content'][:500])  # First 500 chars
    print("...")
    print(text_data['content'][-500:])  # Last 500 chars

TypeError: 'NoneType' object is not subscriptable

In [14]:
db.get_wikipedia_texts()

TypeError: EntityDatabase.get_wikipedia_texts() missing 1 required positional argument: 'entity_id'

In [16]:
from src.data import EntityDatabase

db = EntityDatabase("data/entities.db")

# Get for specific entity

# Get for all entities
cursor = db.conn.cursor()
cursor.execute("""
    SELECT
        e.wikidata_id,
        el.label,
        em.metadata_value as num_langs
    FROM entities e
    LEFT JOIN entity_labels el ON e.id = el.entity_id AND el.language = 'en'
    LEFT JOIN entity_metadata em ON e.id = em.entity_id
        AND em.metadata_key = 'num_wikipedia_languages'
    ORDER BY CAST(em.metadata_value AS INTEGER) DESC
""")

for row in cursor.fetchall():
    print(f"{row['label']}: {row['num_langs']} Wikipedia languages")

db.close()

2025-12-17 00:04:02,920 - src.data.database - INFO - Database initialized at data/entities.db
Francisco Franco: 117 Wikipedia languages
Willy Brandt: 101 Wikipedia languages
Mariano Rajoy: 98 Wikipedia languages
Iker Casillas: 94 Wikipedia languages
Jacinto Benavente: 86 Wikipedia languages
Santiago Ramón y Cajal: 81 Wikipedia languages
Pedro Almodóvar: 81 Wikipedia languages
Montserrat Caballé: 78 Wikipedia languages
José Martí: 74 Wikipedia languages
Manuel Valls: 66 Wikipedia languages
Raúl Albiol: 66 Wikipedia languages
Juan Antonio Samaranch: 64 Wikipedia languages
José María Aznar: 64 Wikipedia languages
Miguel Induráin: 60 Wikipedia languages
Joan Enric Vives Sicília: 58 Wikipedia languages
Amancio Ortega: 53 Wikipedia languages
Julen Lopetegui: 50 Wikipedia languages
Javier Solana: 50 Wikipedia languages
Artur Mas: 49 Wikipedia languages
Carlos Sastre: 48 Wikipedia languages
Camilo Cienfuegos: 48 Wikipedia languages
Lluís Companys: 47 Wikipedia languages
Rafael Alberti: 43 Wiki

In [9]:
#!/usr/bin/env python3
"""Test if SPARQL query with ALL labels works (no ORDER BY)"""

import requests
import time

query = """
SELECT DISTINCT
    ?person
    ?personLabel_en ?personLabel_fr ?personLabel_ar
    ?birthDate
    ?birthPlace
    ?birthPlaceLabel_en ?birthPlaceLabel_fr ?birthPlaceLabel_ar
    ?nationality
    ?nationalityLabel_en ?nationalityLabel_fr ?nationalityLabel_ar
    ?occupation
    ?occupationLabel_en ?occupationLabel_fr ?occupationLabel_ar
WHERE {
  VALUES ?nationality { wd:Q142 wd:Q183 wd:Q30 wd:Q29 }

  ?person wdt:P31 wd:Q5 .
  ?person wdt:P27 ?nationality .
  ?person wdt:P569 ?birthDate .
  ?person wdt:P19 ?birthPlace .
  ?person wdt:P106 ?occupation .

  FILTER(YEAR(?birthDate) > 1850)
  FILTER(YEAR(?birthDate) < 2000)

  # Person labels
  OPTIONAL {
    ?person rdfs:label ?personLabel_en .
    FILTER(LANG(?personLabel_en) = "en")
  }
  OPTIONAL {
    ?person rdfs:label ?personLabel_fr .
    FILTER(LANG(?personLabel_fr) = "fr")
  }
  OPTIONAL {
    ?person rdfs:label ?personLabel_ar .
    FILTER(LANG(?personLabel_ar) = "ar")
  }

  # Birth place labels
  OPTIONAL {
    ?birthPlace rdfs:label ?birthPlaceLabel_en .
    FILTER(LANG(?birthPlaceLabel_en) = "en")
  }
  OPTIONAL {
    ?birthPlace rdfs:label ?birthPlaceLabel_fr .
    FILTER(LANG(?birthPlaceLabel_fr) = "fr")
  }
  OPTIONAL {
    ?birthPlace rdfs:label ?birthPlaceLabel_ar .
    FILTER(LANG(?birthPlaceLabel_ar) = "ar")
  }

  # Nationality labels
  OPTIONAL {
    ?nationality rdfs:label ?nationalityLabel_en .
    FILTER(LANG(?nationalityLabel_en) = "en")
  }
  OPTIONAL {
    ?nationality rdfs:label ?nationalityLabel_fr .
    FILTER(LANG(?nationalityLabel_fr) = "fr")
  }
  OPTIONAL {
    ?nationality rdfs:label ?nationalityLabel_ar .
    FILTER(LANG(?nationalityLabel_ar) = "ar")
  }

  # Occupation labels
  OPTIONAL {
    ?occupation rdfs:label ?occupationLabel_en .
    FILTER(LANG(?occupationLabel_en) = "en")
  }
  OPTIONAL {
    ?occupation rdfs:label ?occupationLabel_fr .
    FILTER(LANG(?occupationLabel_fr) = "fr")
  }
  OPTIONAL {
    ?occupation rdfs:label ?occupationLabel_ar .
    FILTER(LANG(?occupationLabel_ar) = "ar")
  }
}
LIMIT 10000
"""

print("Testing SPARQL query with ALL labels (16 OPTIONAL blocks)...")
print("No ORDER BY - should be fast!")
print("=" * 80)

start = time.time()

try:
    response = requests.get(
        "https://query.wikidata.org/sparql",
        params={'query': query, 'format': 'json'},
        headers={'User-Agent': 'TestBot/1.0'},
        timeout=60
    )

    elapsed = time.time() - start

    if response.status_code == 200:
        data = response.json()
        results = data['results']['bindings']

        print(f"✅ SUCCESS in {elapsed:.2f} seconds!")
        print(f"Got {len(results)} results")

        # Check if labels are present
        if results:
            first = results[0]
            print(f"\nFirst result:")
            print(f"  Person: {first.get('personLabel_en', {}).get('value', 'MISSING')}")
            print(f"  BirthPlace: {first.get('birthPlaceLabel_en', {}).get('value', 'MISSING')}")
            print(f"  Nationality: {first.get('nationalityLabel_en', {}).get('value', 'MISSING')}")
            print(f"  Occupation: {first.get('occupationLabel_en', {}).get('value', 'MISSING')}")

        print("\n" + "=" * 80)
        print("🎉 ALL LABELS IN SPARQL WORKS!")
        print("No need for 30k API calls - everything in one query!")
        print("=" * 80)
    else:
        print(f"❌ FAILED: {response.status_code}")

except requests.Timeout:
    elapsed = time.time() - start
    print(f"❌ TIMEOUT after {elapsed:.2f}s")
    print("Labels in SPARQL still too slow :(")
except Exception as e:
    print(f"❌ ERROR: {e}")

Testing SPARQL query with ALL labels (16 OPTIONAL blocks)...
No ORDER BY - should be fast!
✅ SUCCESS in 5.86 seconds!
Got 10000 results

First result:
  Person: Cordelia Edvardson
  BirthPlace: Munich
  Nationality: Spain
  Occupation: poet

🎉 ALL LABELS IN SPARQL WORKS!
No need for 30k API calls - everything in one query!
